# Reconstruct a signal and evaluate the optional autoencoder

Run in the matching PycWB source checkout. Install TensorFlow separately in
this notebook's Python environment (`python -m pip install tensorflow`). The
classifier uses the bundled historical BLIP autoencoder weights. Its score is
an illustrative model output, not a calibrated false-alarm probability.


In [ ]:
from pathlib import Path
import os
import sys
import numpy as np
import matplotlib.pyplot as plt

# Launch from this source checkout (or its examples directory).
candidates = [Path.cwd(), Path.cwd() / "pycwb", *Path.cwd().parents]
repo = next((p for p in candidates if (p / "examples/demo/user_parameters.yaml").is_file()), None)
if repo is None:
    raise RuntimeError("Open this notebook from the matching PycWB source checkout.")
sys.path.insert(0, str(repo / "examples"))
from notebook_support import prepare_injection_config
from pycwb.modules.logger import logger_init
logger_init()
%matplotlib inline


In [ ]:
# A short, loud CBC exercise; these are educational settings.
# The larger configuration beside this notebook remains available for longer runs.
parameters = dict(approximant="IMRPhenomXHM", mass1=20, mass2=20,
    spin1z=0, spin2z=0, distance=200, inclination=0, coa_phase=0,
    f_lower=20.0, gps_time=1126259526, ra=1.0, dec=0.3, pol=0.8,
    t_start=-8.0, t_end=1.0)
work, config_path, config = prepare_injection_config("native-injection", parameters=parameters)
print(config.ifo, config.rateANA, work)


## Generate seeded noise and inject a waveform


In [ ]:
from pycwb.modules.job_segment import create_job_segment_from_config
from pycwb.modules.read_data.simulations import generate_noise_for_job_seg, generate_injections

job_segment = create_job_segment_from_config(config)[0]
data = generate_noise_for_job_seg(job_segment, config.inRate, f_low=config.fLow)
data = generate_injections(config, job_segment, data)
fig, ax = plt.subplots()
ax.plot(data[0].sample_times, data[0].data)
ax.set(xlabel="GPS time [s]", ylabel="L1 strain")


## Resample, regress and whiten


In [ ]:
from pycwb.modules.read_data.data_check import check_and_resample_py
from pycwb.modules.data_conditioning import condition_strains

data = [check_and_resample_py(strain, config, i) for i, strain in enumerate(data)]
strains, nRMS = condition_strains(config, data)
fig, ax = plt.subplots(figsize=(12, 4))
ax.specgram(strains[0].data, NFFT=512, Fs=float(strains[0].sample_rate), noverlap=384)
ax.set(ylim=(config.fLow, config.fHigh), xlabel="Time from padded segment start [s]", ylabel="Frequency [Hz]")


## Select pixels and form clusters


In [ ]:
from pycwb.modules.coherence_native import coherence

fragment_clusters = coherence(config, strains, job_seg=job_segment, nRMS=nRMS)
print("Fragment clusters:", sum(len(level[0].clusters) for level in fragment_clusters))


## Combine resolutions and apply subnet cuts


In [ ]:
from pycwb.modules.super_cluster_native import supercluster_wrapper
from pycwb.modules.xtalk.monster import load_catalog

xtalk_coeff, xtalk_lookup_table, layers, _ = load_catalog(config.MRAcatalog)
super_fragment_clusters = supercluster_wrapper(
    config, fragment_clusters, strains, xtalk_coeff, xtalk_lookup_table, layers,
    job_seg=job_segment,
)
print("Superclusters:", sum(len(fc.clusters) for fc in (super_fragment_clusters or [])))


## Evaluate likelihood and inspect accepted candidates


In [ ]:
from pycwb.modules.likelihoodWP import evaluate_fragment_clusters
from pycwb.types.network_event import Event

results = evaluate_fragment_clusters(config, super_fragment_clusters or [], strains,
                                     config.MRAcatalog, nRMS=nRMS)
events, clusters = [], []
for lag, lag_results in enumerate(results):
    for cluster, sky_stats in lag_results:
        event = Event()
        event.output_py(job_segment, cluster, config, lag_shifts=job_segment.lag_shifts[lag])
        events.append(event)
        clusters.append(cluster)
        print("rho:", cluster.cluster_meta.net_rho, "net_cc:", cluster.cluster_meta.net_cc)
print("Accepted candidates:", len(events))


## Likelihood pixels and reconstructed waveforms


In [ ]:
from pycwb.modules.reconstruction import get_network_MRA_wave

waves = []
for cluster in clusters:
    values, start, dt, df = cluster.get_sparse_map("likelihood")
    fig, ax = plt.subplots()
    image = ax.imshow(values.T, origin="lower", aspect="auto",
                      extent=(start, start + values.shape[0] * dt, 0, values.shape[1] * df))
    ax.set(xlabel="Time from padded segment start [s]", ylabel="Frequency [Hz]", title="Cluster likelihood map")
    fig.colorbar(image, ax=ax, label="Pixel likelihood")
    recovered = get_network_MRA_wave(config, cluster, config.rateANA, config.nIFO,
                                     config.TDRate, "signal", 0, True, whiten=True,
                                     start_time=job_segment.padded_start)
    waves.append(recovered)
    fig, ax = plt.subplots()
    for ifo, wave in zip(config.ifo, recovered):
        ax.plot(wave.sample_times, wave.data, label=ifo)
    ax.set(xlabel="GPS time [s]", ylabel="Reconstructed whitened strain")
    ax.legend()
if not clusters:
    print("No candidate passed the cuts; inspect the preceding stages.")


These stage calls expose intermediate values. For normal searches use
`pycwb run CONFIG --work-dir RUN`, which also records progress, catalogs,
injection matching and native HDF products. This exercise is not a sensitivity
or significance measurement. The output parent can be changed with
`PYCWB_EXAMPLES_WORK_DIR`; select a fresh parent when rerunning. Set
`PYCWB_EXAMPLES_XTALK` to reuse an existing compatible catalog.


## Evaluate the historical classifier


In [ ]:
from pycwb.modules.autoencoder import get_glitchness

if not waves:
    raise RuntimeError("The classifier requires an accepted reconstructed candidate.")
score = get_glitchness(config, waves[0], events[0].sSNR, events[0].likelihood)
if score is None or not np.isfinite(score):
    raise RuntimeError("Autoencoder evaluation failed; inspect its error message and TensorFlow installation.")
print("Autoencoder score:", score)
